# Why did credits jump last week?
Window: the 7 days before 2026-11-10 (exclusive) against the 7 days before that. Synthetic values; the shapes of the results are real.

In [ ]:
# Connection only. Every value comes from an environment variable; nothing secret is stored here.
import os

import pandas as pd
import snowflake.connector

key_file = os.environ.get("SNOWFLAKE_PRIVATE_KEY_FILE")
auth = ({"authenticator": "SNOWFLAKE_JWT", "private_key_file": key_file} if key_file
        else {"authenticator": os.environ.get("SNOWFLAKE_AUTHENTICATOR", "externalbrowser")})
conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    role=os.environ["SNOWFLAKE_ROLE"],            # a read-only role
    warehouse=os.environ["SNOWFLAKE_WAREHOUSE"],  # X-Small is enough for system views
    session_parameters={"TIMEZONE": "UTC", "QUERY_TAG": "analysis:cost-jump"},
    **auth,
)


def q(sql, params=None):
    """Run one read-only statement and return a DataFrame (SHOW output too)."""
    cur = conn.cursor()
    try:
        cur.execute(sql, params)
        return pd.DataFrame(cur.fetchall(), columns=[c[0] for c in cur.description])
    finally:
        cur.close()

### 0. Can this role see the views the answer needs?
- **What:** counts rows in each view queries 1-4 read, over both weeks.
- **Reads:** METERING_DAILY_HISTORY, METERING_HISTORY, QUERY_ATTRIBUTION_HISTORY, WAREHOUSE_METERING_HISTORY; no join.
- **Look for:** an error or 0 rows = not visible to this role (USAGE_VIEWER for metering, GOVERNANCE_VIEWER for attribution), not "nothing found". Not counted in the 1-4 queries.

In [ ]:
q("""
WITH p AS (SELECT 7 AS days, '2026-11-10'::DATE AS end_date)
SELECT 'METERING_DAILY_HISTORY' AS src, COUNT(*) AS rows_in_window
FROM SNOWFLAKE.ACCOUNT_USAGE.METERING_DAILY_HISTORY, p
WHERE usage_date >= DATEADD(day, -2 * p.days, p.end_date) AND usage_date < p.end_date
UNION ALL
SELECT 'METERING_HISTORY', COUNT(*) FROM SNOWFLAKE.ACCOUNT_USAGE.METERING_HISTORY, p
WHERE start_time >= DATEADD(day, -2 * p.days, p.end_date) AND start_time < p.end_date
UNION ALL
SELECT 'QUERY_ATTRIBUTION_HISTORY', COUNT(*) FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_ATTRIBUTION_HISTORY, p
WHERE start_time >= DATEADD(day, -2 * p.days, p.end_date) AND start_time < p.end_date
UNION ALL
SELECT 'WAREHOUSE_METERING_HISTORY', COUNT(*) FROM SNOWFLAKE.ACCOUNT_USAGE.WAREHOUSE_METERING_HISTORY, p
WHERE start_time >= DATEADD(day, -2 * p.days, p.end_date) AND start_time < p.end_date
""")

                          SRC  ROWS_IN_WINDOW
0      METERING_DAILY_HISTORY              56
1            METERING_HISTORY            1176
2   QUERY_ATTRIBUTION_HISTORY           41230
3  WAREHOUSE_METERING_HISTORY            1008

### 1. Which service moved?
- **What:** billed credits per service type, this week against last week.
- **Reads:** METERING_DAILY_HISTORY (day x service type); no join; billed = after the cloud services adjustment.
- **Look for:** the service with the largest absolute change; `change_pct` NULL = new service. The last day is complete: the window ends at midnight UTC and the run was 10 h later.

In [ ]:
q("""
WITH p AS (SELECT 7 AS days, '2026-11-10'::DATE AS end_date),
d AS (
  SELECT service_type,
         SUM(IFF(usage_date >= DATEADD(day, -p.days, p.end_date), credits_billed, 0)) AS cur_credits,
         SUM(IFF(usage_date <  DATEADD(day, -p.days, p.end_date), credits_billed, 0)) AS prev_credits
  FROM SNOWFLAKE.ACCOUNT_USAGE.METERING_DAILY_HISTORY, p
  WHERE usage_date >= DATEADD(day, -2 * p.days, p.end_date) AND usage_date < p.end_date
  GROUP BY service_type
)
SELECT service_type, prev_credits, cur_credits, cur_credits - prev_credits AS change_credits,
       ROUND(100 * (cur_credits - prev_credits) / NULLIF(prev_credits, 0), 1) AS change_pct
FROM d
ORDER BY ABS(cur_credits - prev_credits) DESC
""")

         SERVICE_TYPE  PREV_CREDITS  CUR_CREDITS  CHANGE_CREDITS  CHANGE_PCT
0  WAREHOUSE_METERING        1210.4       1642.9           432.5        35.7
1     AUTO_CLUSTERING          88.1         91.0             2.9         3.3
2         AI_SERVICES          12.0         14.6             2.6        21.7
3     SERVERLESS_TASK          40.2         39.8            -0.4        -1.0

### 2. Which warehouse moved?
- **What:** credits used per warehouse, this week against last week.
- **Reads:** METERING_HISTORY (hour x service type x entity), service WAREHOUSE_METERING; no join.
- **Look for:** the warehouse holding most of the +432.5 credits. Used credits (before the cloud services adjustment) rank warehouses; they are not the bill.

In [ ]:
q("""
WITH p AS (SELECT 7 AS days, '2026-11-10'::DATE AS end_date)
SELECT entity_id, MAX(name) AS warehouse_name,
       SUM(IFF(start_time >= DATEADD(day, -p.days, p.end_date), credits_used, 0)) AS cur_credits_used,
       SUM(IFF(start_time <  DATEADD(day, -p.days, p.end_date), credits_used, 0)) AS prev_credits_used
FROM SNOWFLAKE.ACCOUNT_USAGE.METERING_HISTORY, p
WHERE service_type = 'WAREHOUSE_METERING'
  AND start_time >= DATEADD(day, -2 * p.days, p.end_date) AND start_time < p.end_date
GROUP BY entity_id
ORDER BY ABS(cur_credits_used - prev_credits_used) DESC
LIMIT 10
""")

   ENTITY_ID  WAREHOUSE_NAME  CUR_CREDITS_USED  PREV_CREDITS_USED
0        104    TRANSFORM_WH            1021.7              610.3
1        101           BI_WH             418.5              402.0
2        107      LOADING_WH             214.9              211.6

### 3. New or grown query shapes on TRANSFORM_WH
- **What:** attributed compute credits per parameterized query shape on one warehouse, this week against last week.
- **Reads:** QUERY_ATTRIBUTION_HISTORY (per query), grouped by QUERY_PARAMETERIZED_HASH; no join.
- **Look for:** `prev_credits = 0` = a new workload. Idle time is not in this view (query 4). The view lags up to 8 h; the window ended 10 h before the run.

In [ ]:
q("""
WITH p AS (SELECT 7 AS days, '2026-11-10'::DATE AS end_date, 'TRANSFORM_WH' AS wh)
SELECT query_parameterized_hash AS shape,
       SUM(IFF(start_time <  DATEADD(day, -p.days, p.end_date), credits_attributed_compute, 0)) AS prev_credits,
       SUM(IFF(start_time >= DATEADD(day, -p.days, p.end_date), credits_attributed_compute, 0)) AS cur_credits,
       COUNT(DISTINCT IFF(start_time >= DATEADD(day, -p.days, p.end_date), query_id, NULL)) AS cur_queries,
       MAX_BY(query_id, credits_attributed_compute) AS costliest_query_id
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_ATTRIBUTION_HISTORY, p
WHERE warehouse_name = p.wh
  AND start_time >= DATEADD(day, -2 * p.days, p.end_date) AND start_time < p.end_date
GROUP BY 1
ORDER BY cur_credits - prev_credits DESC
LIMIT 10
""")

      SHAPE  PREV_CREDITS  CUR_CREDITS  CUR_QUERIES                    COSTLIEST_QUERY_ID
0  9f3a1c07           0.0        268.4          168  01b8e2a4-0000-4c1d-0000-000000000001
1  41bc77d2         120.2        131.0           56  01b8e2a4-0000-4c1d-0000-000000000002
2  07de55a9          96.4         98.1          336  01b8e2a4-0000-4c1d-0000-000000000003

### 4. Idle credits on TRANSFORM_WH
- **What:** compute credits minus credits attributed to queries, per week, for hours older than the 8 h attribution lag.
- **Reads:** WAREHOUSE_METERING_HISTORY (hour x warehouse); no join.
- **Look for:** the rest of the warehouse's increase. NULL attribution (adaptive warehouse) = not assessed, not zero idle.

In [ ]:
q("""
WITH p AS (SELECT 7 AS days, '2026-11-10'::DATE AS end_date, 'TRANSFORM_WH' AS wh)
SELECT IFF(start_time >= DATEADD(day, -p.days, p.end_date), 'current', 'previous') AS week,
       SUM(credits_used_compute) AS compute_credits,
       SUM(credits_attributed_compute_queries) AS attributed_credits,
       SUM(credits_used_compute) - SUM(credits_attributed_compute_queries) AS idle_credits
FROM SNOWFLAKE.ACCOUNT_USAGE.WAREHOUSE_METERING_HISTORY, p
WHERE warehouse_name = p.wh AND credits_attributed_compute_queries IS NOT NULL
  AND start_time >= DATEADD(day, -2 * p.days, p.end_date) AND start_time < p.end_date
  AND start_time < DATEADD(hour, -8, CURRENT_TIMESTAMP())
GROUP BY 1
ORDER BY 1
""")

       WEEK  COMPUTE_CREDITS  ATTRIBUTED_CREDITS  IDLE_CREDITS
0   current            998.2               751.9         246.3
1  previous            594.7               448.6         146.1

## Findings
- Confirmed: the week's +432.5 billed credits are warehouse compute (query 1); TRANSFORM_WH holds +411.4 of the used credits (query 2).
- Confirmed: a new query shape `9f3a1c07` started this week on TRANSFORM_WH: 168 runs, 268.4 attributed credits (query 3).
- Confirmed: idle credits on TRANSFORM_WH rose by 100.2 (query 4); the new runs keep the warehouse awake between batches.
- Not verified: which job sends shape `9f3a1c07`; its queries carry no query tag. Next query: match `costliest_query_id` against `TASK_HISTORY.QUERY_ID` and `QUERY_HISTORY.USER_NAME`.